# 06 · Judge Evals: when a rule is not enough

Notebook 05 checked things we can test exactly: was a refund issued, is a word present. But how do you test "is this reply **polite and helpful**"? There is no exact rule.

One answer is **LLM-as-judge**: another model grades the reply against a rubric. It is flexible, but it is also a model, so it can be wrong. We have to **check the judge** too.

In [1]:
import shop
import guards
from shop import client, MODEL

print("Ready")

Ready


## The judge
It sees the customer's email, the reply and the evidence (what the tools returned), and gives a score from 1 to 5 with a short reason. The rubric says what each score means. As before, `reasoning` comes first.

In [2]:
from pydantic import BaseModel

class Grade(BaseModel):
    reasoning: str    # first, one short sentence
    score: int        # 1 (bad) to 5 (excellent)

def judge_reply(customer_email, reply, evidence):
    response = client.chat.completions.parse(
        model=guards.REVIEWER_MODEL,      # a stronger model than the one that wrote the replies
        messages=[
            {"role": "system", "content": (
                "You grade replies from a customer support agent. Give a score from 1 to 5.\n"
                "5 = accurate to the evidence, polite, clear and helpful.\n"
                "3 = accurate but cold, wordy or not very helpful.\n"
                "1 = wrong facts, rude, or confusing.\n"
                "Judge only against the EVIDENCE, not against what you think is true."
            )},
            {"role": "user", "content": "CUSTOMER EMAIL:\n" + customer_email + "\n\nEVIDENCE:\n" + evidence + "\n\nREPLY TO GRADE:\n" + reply},
        ],
        response_format=Grade,
    )
    return response.choices[0].message.parsed

## Can we trust the judge?
We cannot just assume so. We write six replies **and score them ourselves first**. These are our "human labels". Then we see how close the judge gets.

In [3]:
customer_email = "Hi, where is my order A1002?"
evidence = shop.lookup_order("A1002")
date = shop.ORDERS["A1002"]["expected_delivery"]

# (what the reply says, the score a human would give)
labelled_replies = [
    ("Your order A1002 (running shoes) is in transit and expected on " + date + ". Thanks for your patience!", 5),
    ("Hello! Your running shoes are on their way and should arrive on " + date + ". Anything else I can help with?", 5),
    ("It's in transit. Expected " + date + ".", 3),
    ("Your order A1002 is in transit. We are very sorry for any inconvenience this may have caused and we truly appreciate your patience and loyalty as a valued customer of our shop.", 3),
    ("Your order was delivered yesterday.", 1),
    ("Why do you keep asking? Check the tracking link yourself.", 1),
]

In [4]:
exact = 0
close = 0

for reply, human_score in labelled_replies:
    grade = judge_reply(customer_email, reply, evidence)

    if grade.score == human_score:
        exact = exact + 1
    if abs(grade.score - human_score) <= 1:
        close = close + 1

    print("Human:", human_score, " Judge:", grade.score, "|", reply[:60])
    print("      judge says:", grade.reasoning)

print()
print("Exact agreement      :", exact, "of", len(labelled_replies))
print("Within 1 point       :", close, "of", len(labelled_replies))

Human: 5  Judge: 5 | Your order A1002 (running shoes) is in transit and expected 
      judge says: The reply accurately reflects the evidence provided about the order status and expected delivery date. It is clear and uses a polite tone, appropriately thanking the customer for their patience. The information is precise and directly relevant to the customer's question.


Human: 5  Judge: 5 | Hello! Your running shoes are on their way and should arrive
      judge says: The response accurately states that the order (running shoes) is on its way and provides the correct expected delivery date of 2026-11-25, matching the evidence given. The tone is polite, clear, and helpful as it offers further assistance to the customer.


Human: 3  Judge: 3 | It's in transit. Expected 2026-11-25.
      judge says: The reply accurately states the status ("in transit") and expected delivery date ("2026-11-25") of the order based on the provided evidence. However, it is quite curt and lacks politeness or additional helpful context. A more complete response would offer a greeting, expression of understanding or sympathy for the customer's concern, and an invitation for further questions.


Human: 3  Judge: 4 | Your order A1002 is in transit. We are very sorry for any in
      judge says: The response accurately states that the order is in transit, matching the evidence provided. The tone is polite and expresses appreciation for the customer's patience, which is helpful. However, it could have been more helpful by including the expected delivery date, which is present in the evidence, to provide the customer with more detailed information about their order.


Human: 1  Judge: 1 | Your order was delivered yesterday.
      judge says: The response incorrectly states that the order was delivered yesterday, but the evidence indicates that the order is still in transit, with an expected delivery date in the future (2026-11-25). This constitutes providing wrong information to the customer, which is misleading and potentially harmful to the customer's understanding of their order status.


Human: 1  Judge: 1 | Why do you keep asking? Check the tracking link yourself.
      judge says: The reply is rude and dismisses the customer's request for assistance. It tells the customer to check the tracking link without providing it or explaining where to find it. Additionally, it lacks any helpful context about the status of the order, expected delivery, or any form of polite customer service language.

Exact agreement      : 5 of 6
Within 1 point       : 6 of 6


Look at where the judge **disagrees** with us. A judge that is close on clear cases (very good, very bad) and wobbly on the middle ones is common.

## Is the judge consistent?
The same judge, the same reply, three times:

In [5]:
reply = "It's in transit. Expected " + date + "."

for attempt in range(3):
    grade = judge_reply(customer_email, reply, evidence)
    print("Attempt", attempt + 1, "-> score", grade.score)

Attempt 1 -> score 3


Attempt 2 -> score 3


Attempt 3 -> score 3


## When to use a judge, and when not to

| Use | Example |
|---|---|
| **Code check** (best, whenever possible) | A refund exists in the data, a secret is absent, the status is correct |
| **Judge** (for qualities with no exact rule) | Tone, clarity, helpfulness |

Good habits with judges:
* **Write a clear rubric**, saying what each score means.
* **Check the judge against human labels**, as we just did, and re-check whenever you change it.
* **Use a stronger model for the judge** than for the agent.
* **Beware of bias.** Judges can favour longer answers or answers written by a model like themselves.
* **Use scores as a signal, not as truth.** Track the average over many cases, and read the low ones yourself.

## Recap
* Some qualities cannot be tested by a rule. A model can grade them against a rubric.
* A judge is another model, so **test the judge** against a small set of human-scored examples.
* Prefer code checks. Use a judge only where you must.

You now have layered guardrails and a way to measure them. The real-life script `guarded_support_desk.py` shows them working together, and `run_evals.py` is the test suite you run before every release.